In [3]:
from pathlib import Path
import random
import shutil
import csv


# ============================================================
# 1. CONFIGURACIÓN
# ============================================================

# Cambia esta ruta por la ruta real de tu dataset
DATASET_ORIGINAL = Path(r"C:\MachinL\ProyectoFinal\Clean_dataset")

# La nueva carpeta se creará al lado de Clean_dataset
CARPETA_SALIDA = (
    DATASET_ORIGINAL.parent /
    "Orange_segmentation_manual"
)

CLASES = [
    "Blackspot",
    "Canker",
    "Healthy",
    "HLB",
    "Melanosis"
]

CANTIDADES = {
    "Train": 20,
    "Val": 5,
    "Test": 5
}

EXTENSIONES = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".tif",
    ".tiff",
    ".webp"
}

SEMILLA = 42

# False: detiene el código si la carpeta ya existe.
# True: elimina la carpeta anterior y la vuelve a crear.
SOBRESCRIBIR = False

In [4]:
# ============================================================
# 2. VALIDAR LAS CARPETAS
# ============================================================

if not DATASET_ORIGINAL.exists():
    raise FileNotFoundError(
        f"No se encontró el dataset:\n{DATASET_ORIGINAL}"
    )

if CARPETA_SALIDA.exists():
    if SOBRESCRIBIR:
        shutil.rmtree(CARPETA_SALIDA)
        print("Se eliminó la carpeta de salida anterior.")
    else:
        raise FileExistsError(
            f"La carpeta de salida ya existe:\n"
            f"{CARPETA_SALIDA}\n\n"
            f"Cambia SOBRESCRIBIR = True para volver a crearla."
        )

CARPETA_SALIDA.mkdir(
    parents=True,
    exist_ok=True
)

print("Dataset original:")
print(DATASET_ORIGINAL)

print("\nNueva carpeta:")
print(CARPETA_SALIDA)

Dataset original:
C:\MachinL\ProyectoFinal\Clean_dataset

Nueva carpeta:
C:\MachinL\ProyectoFinal\Orange_segmentation_manual


In [5]:
# ============================================================
# 3. OBTENER LAS IMÁGENES DE UNA CARPETA
# ============================================================

def obtener_imagenes(carpeta):
    return sorted([
        archivo
        for archivo in carpeta.iterdir()
        if (
            archivo.is_file()
            and archivo.suffix.lower() in EXTENSIONES
        )
    ])

In [6]:
# ============================================================
# 4. SELECCIONAR Y COPIAR LAS IMÁGENES
# ============================================================

random.seed(SEMILLA)

registros = []

for split, cantidad_por_clase in CANTIDADES.items():

    print(f"\nProcesando {split}")

    for clase in CLASES:

        carpeta_origen = (
            DATASET_ORIGINAL /
            split /
            clase
        )

        if not carpeta_origen.exists():
            raise FileNotFoundError(
                f"No existe la carpeta:\n{carpeta_origen}"
            )

        imagenes = obtener_imagenes(
            carpeta_origen
        )

        if len(imagenes) < cantidad_por_clase:
            raise ValueError(
                f"No hay suficientes imágenes en:\n"
                f"{carpeta_origen}\n"
                f"Disponibles: {len(imagenes)}\n"
                f"Solicitadas: {cantidad_por_clase}"
            )

        seleccionadas = random.sample(
            imagenes,
            cantidad_por_clase
        )

        carpeta_destino = (
            CARPETA_SALIDA /
            split /
            clase
        )

        carpeta_destino.mkdir(
            parents=True,
            exist_ok=True
        )

        for imagen_origen in seleccionadas:

            imagen_destino = (
                carpeta_destino /
                imagen_origen.name
            )

            shutil.copy2(
                imagen_origen,
                imagen_destino
            )

            registros.append({
                "split": split,
                "clase": clase,
                "nombre_archivo": imagen_origen.name,
                "ruta_original": str(imagen_origen),
                "ruta_copiada": str(imagen_destino)
            })

        print(
            f"  {clase:<12}: "
            f"{len(seleccionadas)} imágenes"
        )


Procesando Train
  Blackspot   : 20 imágenes
  Canker      : 20 imágenes
  Healthy     : 20 imágenes
  HLB         : 20 imágenes
  Melanosis   : 20 imágenes

Procesando Val
  Blackspot   : 5 imágenes
  Canker      : 5 imágenes
  Healthy     : 5 imágenes
  HLB         : 5 imágenes
  Melanosis   : 5 imágenes

Procesando Test
  Blackspot   : 5 imágenes
  Canker      : 5 imágenes
  Healthy     : 5 imágenes
  HLB         : 5 imágenes
  Melanosis   : 5 imágenes


In [7]:
# ============================================================
# 5. GUARDAR EL REGISTRO EN CSV
# ============================================================

ruta_csv = (
    CARPETA_SALIDA /
    "seleccion_imagenes.csv"
)

with open(
    ruta_csv,
    "w",
    newline="",
    encoding="utf-8-sig"
) as archivo_csv:

    columnas = [
        "split",
        "clase",
        "nombre_archivo",
        "ruta_original",
        "ruta_copiada"
    ]

    escritor = csv.DictWriter(
        archivo_csv,
        fieldnames=columnas
    )

    escritor.writeheader()
    escritor.writerows(registros)

print("\nArchivo CSV creado:")
print(ruta_csv)


Archivo CSV creado:
C:\MachinL\ProyectoFinal\Orange_segmentation_manual\seleccion_imagenes.csv


In [ ]:
# ============================================================
# 6. VERIFICAR LOS CONTEOS
# ============================================================

total_general = 0

print("\n====================================")
print("RESUMEN DE LA SELECCIÓN")
print("====================================")

for split in CANTIDADES:

    total_split = 0

    print(f"\n{split}")

    for clase in CLASES:

        carpeta = (
            CARPETA_SALIDA /
            split /
            clase
        )

        cantidad = len(
            obtener_imagenes(carpeta)
        )

        total_split += cantidad
        total_general += cantidad

        print(
            f"  {clase:<12}: {cantidad}"
        )

    print(
        f"  Total {split}: {total_split}"
    )

print("\nTotal general:", total_general)

if total_general == 150:
    print("La selección contiene las 150 imágenes esperadas.")
else:
    print(
        "Revisa los conteos porque el total no es 150."
    )